# Mount Drive

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Import libraries

In [24]:
import pandas as pd
import numpy as np

# Read dataset

In [25]:
df = pd.read_csv('/content/drive/MyDrive/ML/computational data mining/تمرین هفتم/sklearn-anatomy.csv')
df

,page_url,page_title,page_length,link_text,link_href,link_url
0,https://scikit-learn.org/stable/index.html,scikit-learn: machine learning in Python — sci...,27767,Install,install.html,https://scikit-learn.org/stable/install.html
1,https://scikit-learn.org/stable/index.html,scikit-learn: machine learning in Python — sci...,27767,User Guide,user_guide.html,https://scikit-learn.org/stable/user_guide.html
2,https://scikit-learn.org/stable/index.html,scikit-learn: machine learning in Python — sci...,27767,API,modules/classes.html,https://scikit-learn.org/stable/modules/classe...
3,https://scikit-learn.org/stable/index.html,scikit-learn: machine learning in Python — sci...,27767,Examples,auto_examples/index.html,https://scikit-learn.org/stable/auto_examples/...
4,https://scikit-learn.org/stable/index.html,scikit-learn: machine learning in Python — sci...,27767,Community,https://blog.scikit-learn.org/,https://blog.scikit-learn.org/
...,...,...,...,...,...,...
62197,https://scikit-learn.org/stable/whats_new/olde...,Version 0.12.1 — scikit-learn 1.1.1 documentation,134884,documentation for the SVM module,http://scikit-learn.org/stable/modules/svm.html,http://scikit-learn.org/stable/modules/svm.html
62198,https://scikit-learn.org/stable/whats_new/olde...,Version 0.12.1 — scikit-learn 1.1.1 documentation,134884,class reference,http://scikit-learn.org/stable/modules/classes...,http://scikit-learn.org/stable/modules/classes...
62199,https://scikit-learn.org/stable/whats_new/olde...,Version 0.12.1 — scikit-learn 1.1.1 documentation,134884,Classification of text documents using sparse ...,../auto_examples/text/plot_document_classifica...,https://scikit-learn.org/stable/auto_examples/...
62200,https://scikit-learn.org/stable/whats_new/olde...,Version 0.12.1 — scikit-learn 1.1.1 documentation,134884,See here,http://scikit-learn.org/stable/auto_examples/i...,http://scikit-learn.org/stable/auto_examples/i...


# PageRank Algorithm

## Select Relevant Columns

We only keep `page_url` and `link_url` because they are needed to build the directed graph.

- `page_url`: the source page.
- `link_url`: the page linked from the source page.

In [26]:
df = df[['page_url', 'link_url']]
df

,page_url,link_url
0,https://scikit-learn.org/stable/index.html,https://scikit-learn.org/stable/install.html
1,https://scikit-learn.org/stable/index.html,https://scikit-learn.org/stable/user_guide.html
2,https://scikit-learn.org/stable/index.html,https://scikit-learn.org/stable/modules/classe...
3,https://scikit-learn.org/stable/index.html,https://scikit-learn.org/stable/auto_examples/...
4,https://scikit-learn.org/stable/index.html,https://blog.scikit-learn.org/
...,...,...
62197,https://scikit-learn.org/stable/whats_new/olde...,http://scikit-learn.org/stable/modules/svm.html
62198,https://scikit-learn.org/stable/whats_new/olde...,http://scikit-learn.org/stable/modules/classes...
62199,https://scikit-learn.org/stable/whats_new/olde...,https://scikit-learn.org/stable/auto_examples/...
62200,https://scikit-learn.org/stable/whats_new/olde...,http://scikit-learn.org/stable/auto_examples/i...


## Create the Set of Nodes

We combine all source and destination URLs and remove duplicates.

Each unique URL represents one **node (web page)** in the graph.

In [27]:
nodes = pd.concat([df['page_url'], df['link_url']]).unique()
print(nodes.shape)
nodes

(6026,)


array(['https://scikit-learn.org/stable/index.html',
       'https://scikit-learn.org/stable/install.html',
       'https://scikit-learn.org/stable/user_guide.html', ...,
       'http://scikit-learn.org/stable/modules/classes.html',
       'http://scikit-learn.org/stable/auto_examples/index.html',
       'https://scikit-learn.org/stable/_sources/whats_new/older_versions.rst.txt'],
      dtype=object)

## Map URLs to Indices

Each URL is assigned a unique integer index.

This mapping allows us to represent web pages as rows and columns in numerical matrices.

In [28]:
url_to_index = {}
for i in range(len(nodes)):
  url_to_index[nodes[i]] = i
print(len(url_to_index))

6026


## Build the Directed Graph

We create a graph where each URL is a node and each hyperlink is a directed edge.

For each source page, we store all pages it links to. This graph will later be used to build the PageRank transition matrix.

In [29]:
graph = {}
for index, row in df.iterrows():
  if row['page_url'] not in graph:
    graph[row['page_url']] = [row['link_url']]
  else:
    graph[row['page_url']].append(row['link_url'])

print(len(graph))
print(list(graph.items())[:3])

935
[('https://scikit-learn.org/stable/index.html', ['https://scikit-learn.org/stable/install.html', 'https://scikit-learn.org/stable/user_guide.html', 'https://scikit-learn.org/stable/modules/classes.html', 'https://scikit-learn.org/stable/auto_examples/index.html', 'https://blog.scikit-learn.org/', 'https://scikit-learn.org/stable/getting_started.html', 'https://scikit-learn.org/stable/tutorial/index.html', 'https://scikit-learn.org/stable/whats_new/v1.1.html', 'https://scikit-learn.org/stable/glossary.html', 'https://scikit-learn.org/dev/developers/index.html', 'https://scikit-learn.org/stable/faq.html', 'https://scikit-learn.org/stable/support.html', 'https://scikit-learn.org/stable/related_projects.html', 'https://scikit-learn.org/stable/roadmap.html', 'https://scikit-learn.org/stable/about.html', 'https://github.com/scikit-learn/scikit-learn', 'https://scikit-learn.org/dev/versions.html', 'https://scikit-learn.org/stable/getting_started.html', 'https://scikit-learn.org/stable/tut

## Remove Duplicate Links

Some pages may contain repeated links to the same URL. We remove these duplicates so that each link is counted only once when calculating transition probabilities.

In [30]:
for key in graph:
  graph[key] = list(set(graph[key]))
print(list(graph.items())[:3])

[('https://scikit-learn.org/stable/index.html', ['https://scikit-learn.org/stable/modules/svm.html', 'https://scikit-learn.org/stable/whats_new/v0.22.html', 'https://scikit-learn.org/stable/modules/neighbors.html', 'https://scikit-learn.org/dev/developers/index.html', 'https://scikit-learn.org/dev/versions.html', 'https://scikit-learn.org/stable/tutorial/index.html', 'https://scikit-learn.org/stable/whats_new/v0.24.html', 'https://github.com/scikit-learn/scikit-learn', 'https://blog.scikit-learn.org', 'https://scikit-learn.org/stable/getting_started.html', 'https://www.tiktok.com/@scikit.learn', 'https://mail.python.org/mailman/listinfo/scikit-learn', 'https://scikit-learn.org/stable/model_selection.html', 'https://scikit-learn.org/stable/testimonials/testimonials.html', 'https://scikit-learn.org/stable/modules/ensemble.html', 'https://scikit-learn.org/stable/modules/clustering.html', 'https://scikit-learn.org/stable/whats_new/v0.23.html', 'https://scikit-learn.org/stable/auto_examples

## Construct the Transition Matrix Q

The matrix `Q` represents the probability of moving between web pages by following links.

For a page with `N` outgoing links, each linked page has a transition probability of `1/N`.

`Q[destination, source]` stores this probability.

In [31]:
# make matrix Q
Q = np.zeros((len(nodes), len(nodes)))

for key, value in graph.items():
  number_of_value = len(value)
  for val in value:
    source = url_to_index[key]
    destination = url_to_index[val]
    # Q[destination, source] = probability
    Q[destination, source] = 1/number_of_value

print(Q.shape)
print(Q[:5, :5])

(6026, 6026)
[[0.         0.         0.         0.         0.        ]
 [0.01886792 0.         0.01315789 0.00166945 0.00330033]
 [0.01886792 0.02777778 0.         0.00166945 0.00330033]
 [0.01886792 0.02777778 0.01315789 0.         0.00330033]
 [0.01886792 0.02777778 0.01315789 0.00166945 0.        ]]


## Check the Transition Matrix

We check the sum of each column to verify that the transition probabilities are correctly normalized.

For pages with outgoing links, each column should sum to approximately `1`.

In [32]:
print(sum(Q[:, 0]))
print(sum(Q[:, 1]))
print(sum(Q[:, 2]))
print(sum(Q[:, 3]))
print(sum(Q[:, 4]))
print(sum(Q[:, 5]))
print(sum(Q[:, 6]))
print(sum(Q[:, 7]))
print(sum(Q[:, 8]))
print(sum(Q[:, 9]))
print(sum(Q[:, 10]))

0.9999999999999999
1.0000000000000002
1.000000000000001
0.9999999999999963
1.0000000000000053
0.9999999999999999
0.9999999999999993
1.0000000000000042
0.9999999999999987
1.0000000000000002
0.9999999999999993


## Handle Dangling Nodes and Construct P

A **dangling node** is a page with no outgoing links, so its column in `Q` sums to `0`.

To keep `P` a valid probability matrix, we replace such a column with a uniform probability distribution over all pages.

Thus, each column of `P` sums to approximately `1`.

In [33]:
# Make matrix P
P = Q.copy()
for i in range(P.shape[1]):
  if sum(P[:, i]) == 0:
    P[:, i] = (1 / (P.shape[1]))


print(P.shape)
print(sum(P[:,0]))
print(P[:5, :5])


(6026, 6026)
0.9999999999999999
[[0.         0.         0.         0.         0.        ]
 [0.01886792 0.         0.01315789 0.00166945 0.00330033]
 [0.01886792 0.02777778 0.         0.00166945 0.00330033]
 [0.01886792 0.02777778 0.01315789 0.         0.00330033]
 [0.01886792 0.02777778 0.01315789 0.00166945 0.        ]]


## Verify Matrix P

We check the column sums again after handling dangling nodes.

Each column of `P` should now sum to approximately `1`, confirming that `P` is a valid transition probability matrix.

In [34]:
print(sum(P[:, 0]))
print(sum(P[:, 1]))
print(sum(P[:, 2]))
print(sum(P[:, 3]))
print(sum(P[:, 4]))
print(sum(P[:, 5]))
print(sum(P[:, 6]))
print(sum(P[:, 7]))
print(sum(P[:, 8]))
print(sum(P[:, 9]))
print(sum(P[:, 10]))

0.9999999999999999
1.0000000000000002
1.000000000000001
0.9999999999999963
1.0000000000000053
0.9999999999999999
0.9999999999999993
1.0000000000000042
0.9999999999999987
1.0000000000000002
0.9999999999999993


## Construct the Google Matrix A

We combine the transition matrix `P` with a random teleportation component.

$$
A = \alpha P + (1-\alpha)\frac{1}{n}E
$$

where `α = 0.85` controls the probability of following links, while `1 - α` represents randomly jumping to any page.

In [35]:
# Make matrix A with alfa = 0.85
alpha = 0.85
n = len(P)    # or len(Q)
A = alpha * P + (1 - alpha) * (1 / n) * np.ones((n, n))
print(A.shape)
print(A[:5, :5])

(6026, 6026)
[[2.48921341e-05 2.48921341e-05 2.48921341e-05 2.48921341e-05
  2.48921341e-05]
 [1.60626280e-02 2.48921341e-05 1.12091027e-02 1.44392385e-03
  2.83017266e-03]
 [1.60626280e-02 2.36360032e-02 2.48921341e-05 1.44392385e-03
  2.83017266e-03]
 [1.60626280e-02 2.36360032e-02 1.12091027e-02 2.48921341e-05
  2.83017266e-03]
 [1.60626280e-02 2.36360032e-02 1.12091027e-02 1.44392385e-03
  2.48921341e-05]]


## Verify the Google Matrix

We check the column sums of `A` to make sure it remains a valid transition probability matrix.

Each column should sum to approximately `1`.

In [36]:
print(sum(A[:, 0]))
print(sum(A[:, 1]))
print(sum(A[:, 2]))
print(sum(A[:, 3]))
print(sum(A[:, 4]))
print(sum(A[:, 5]))
print(sum(A[:, 6]))
print(sum(A[:, 7]))
print(sum(A[:, 8]))
print(sum(A[:, 9]))
print(sum(A[:, 10]))

1.000000000000008
1.0000000000000073
1.000000000000007
0.9999999999999957
1.0000000000000109
1.0000000000000064
1.000000000000007
1.0000000000000129
1.0000000000000047
1.0000000000000075
1.0000000000000064


## Compute PageRank Using Power Iteration

We start with an equal PageRank value for every page and repeatedly update the ranking vector by multiplying it by `A`.

After each iteration, the vector is normalized using the L1 norm.

The process stops when the difference between two consecutive ranking vectors becomes smaller than `10⁻⁶`, meaning the PageRank values have converged.

In [37]:
# final algorithm
# Power Iteration

n = len(P)     # or: len(Q)  or n = P.shape[0]
r0 = np.full((n,1),1/n)     # or: r0 = (1/n) * np.ones((n, 1))

qk = np.dot(A, r0)
rk = qk /  np.linalg.norm(qk, ord=1)

i = 0
while True:
  qk = np.dot(A, r0)
  rk = qk /  np.linalg.norm(qk, ord=1)
  i += 1
  if np.linalg.norm(rk - r0, 1) <= (10 ** -6):
    break
  r0 = rk.copy()


print('iteration: ', i)
print(rk[:5])
print(sum(rk))

iteration:  15
[[0.00012881]
 [0.00589627]
 [0.00596873]
 [0.00602693]
 [0.0060186 ]]
[1.]


## Create the Final Results DataFrame

We combine each page URL with its corresponding PageRank score in a DataFrame.

This makes the final rankings easier to sort, inspect, and analyze.

In [38]:
# Create the pandas DataFrame
# initialize data of lists.
data = {'Pages': nodes,
        'Scores': rk.flatten()}

df = pd.DataFrame(data, columns=['Pages', 'Scores'])
df

,Pages,Scores
0,https://scikit-learn.org/stable/index.html,0.000129
1,https://scikit-learn.org/stable/install.html,0.005896
2,https://scikit-learn.org/stable/user_guide.html,0.005969
3,https://scikit-learn.org/stable/modules/classe...,0.006027
4,https://scikit-learn.org/stable/auto_examples/...,0.006019
...,...,...
6021,https://www.ee.columbia.edu/~ronw/,0.000130
6022,http://scikit-learn.org/stable/modules/svm.html,0.000130
6023,http://scikit-learn.org/stable/modules/classes...,0.000130
6024,http://scikit-learn.org/stable/auto_examples/i...,0.000130


## Sort Pages by PageRank Score

We sort the pages by their PageRank scores in descending order.

This places the pages with the highest PageRank at the top of the DataFrame.

In [39]:
# Sort the DataFrame by the 'Scores' column in ascending order
df.sort_values("Scores", axis=0, ascending=False, inplace=True)
df

,Pages,Scores
938,https://scikit-learn.org/dev/versions.html,0.006035
935,https://blog.scikit-learn.org/,0.006035
936,https://scikit-learn.org/dev/developers/index....,0.006035
937,https://github.com/scikit-learn/scikit-learn,0.006035
3,https://scikit-learn.org/stable/modules/classe...,0.006027
...,...,...
5348,https://github.com/scikit-learn/scikit-learn/i...,0.000129
5347,https://github.com/brentyi,0.000129
5346,https://github.com/scikit-learn/scikit-learn/i...,0.000129
5344,https://github.com/nsheth12,0.000129


## Inspect PageRank Results

We inspect the final ranking in three ways:

- **Highest-ranked page:** the first row contains the page with the highest PageRank score.
- **Lowest-ranked page:** the last row contains the page with the lowest PageRank score.
- **Specific page:** we can search for a specific URL and retrieve its PageRank score.

In [40]:
# Get the first row using iloc
first_ranked_page = df.iloc[0]
first_ranked_page

,938
Pages,https://scikit-learn.org/dev/versions.html
Scores,0.006035


In [41]:
# Get the last row using iloc
last_ranked_page = df.iloc[-1]
last_ranked_page

,0
Pages,https://scikit-learn.org/stable/index.html
Scores,0.000129


In [42]:
new_url = df[df['Pages'] == 'https://scikit-learn.org/stable/faq.html']
new_url

,Pages,Scores
9,https://scikit-learn.org/stable/faq.html,0.005944
